<a href="https://colab.research.google.com/github/DuongDacMinh/FinalProject-SweetCheckout/blob/main/Sweetcheckout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio tensorflow pillow opencv-python

import os
import shutil
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models
from PIL import Image, ImageDraw, ImageFont
from google.colab import files
import gradio as gr
import cv2
import matplotlib.pyplot as plt

In [ ]:
GIA_BANH = {
  "Egg_Tart": 21000,
  "Croissant": 30000,
  "Cookies_dua": 23000,
  "Cha_bong_cay": 27000,
  "Patechaud": 30000,
  "Banh_mi_dua_luoi": 15000,
  "Banh_da_lon": 23000,
  "Banh_mi_bo": 18000,
  "Muffin_Viet_Quat": 25000,
  "Banh_chuoi_nuong": 19000
}

for ten, gia in GIA_BANH.items():
    print(f'{ten:20s} : {gia:,} VNĐ')

Egg_Tart             : 21,000 VNĐ
Croissant            : 30,000 VNĐ
Cookies_dua          : 23,000 VNĐ
Cha_bong_cay         : 27,000 VNĐ
Patechaud            : 30,000 VNĐ
Banh_mi_dua_luoi     : 15,000 VNĐ
Banh_da_lon          : 23,000 VNĐ
Banh_mi_bo           : 18,000 VNĐ
Muffin_Viet_Quat     : 25,000 VNĐ
Banh_chuoi_nuong     : 19,000 VNĐ


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

# Tốc độ đọc từ local RAM nhanh gấp 100 lần đọc từ Drive
print("Đang copy dữ liệu vào bộ nhớ máy (RAM)... Đợi 30 giây!")
if os.path.exists('/content/data_tmp'):
    shutil.rmtree('/content/data_tmp')
shutil.copytree('/content/drive/MyDrive/FINAL_AI', '/content/data_tmp')

# Cập nhật đường dẫn train trỏ vào bộ nhớ máy
train_dir = '/content/data_tmp'
print("Copy xong! Giờ bật lại GPU và Train, tốc độ sẽ cực nhanh.")

Đang copy dữ liệu vào bộ nhớ máy (RAM)... Đợi 30 giây!
Copy xong! Giờ bật lại GPU và Train, tốc độ sẽ cực nhanh.


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 1. Trỏ vào thư mục Drive
train_dir = '/content/drive/MyDrive/FINAL_AI'
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Nạp data về chuẩn kích thước 128x128 ban đầu
train_generator = datagen.flow_from_directory(
    train_dir, target_size=(128, 128), batch_size=32, class_mode='categorical', subset='training'
)
val_generator = datagen.flow_from_directory(
    train_dir, target_size=(128, 128), batch_size=32, class_mode='categorical', subset='validation'
)

# 3. Kiến trúc CNN tự xây cải tiến cực mạnh (Thêm BN để chống nhiễu khay inox)
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])

# 4. Lưu đè vào file model thi của bạn
checkpoint = ModelCheckpoint('/content/drive/MyDrive/FINAL_AI/mo_hinh_banh_3I.keras', monitor='val_accuracy', save_best_only=True, mode='max')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("🚀 BẮT ĐẦU TRAIN LẠI MẠNG CNN TỰ XÂY...")
model.fit(train_generator, validation_data=val_generator, epochs=25, callbacks=[checkpoint, early_stop])
print("✅ XONG! Đã lưu model tự xây chuẩn vào Drive.")

Found 1359 images belonging to 10 classes.
Found 338 images belonging to 10 classes.
🚀 BẮT ĐẦU TRAIN LẠI MẠNG CNN TỰ XÂY...
Epoch 1/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 23s 342ms/step - accuracy: 0.4702 - loss: 1.8583 - val_accuracy: 0.1657 - val_loss: 10.1949
Epoch 2/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 7s 170ms/step - accuracy: 0.6343 - loss: 1.1852 - val_accuracy: 0.1657 - val_loss: 17.0165
Epoch 3/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 8s 192ms/step - accuracy: 0.6895 - loss: 0.9498 - val_accuracy: 0.2189 - val_loss: 13.6958
Epoch 4/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 7s 159ms/step - accuracy: 0.7491 - loss: 0.7434 - val_accuracy: 0.0888 - val_loss: 9.8456
Epoch 5/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 8s 194ms/step - accuracy: 0.8175 - loss: 0.5581 - val_accuracy: 0.0680 - val_loss: 5.7248
Epoch 6/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 8s 179ms/step - accuracy: 0.8815 - loss: 0.3939 - val_accuracy: 0.0917 - val_loss: 4.8420
Epoch 7/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 7s 151ms/step - accuracy: 0.9036 - loss: 0.3092 - val_accuracy: 0.

In [ ]:
model.save('/content/drive/MyDrive/FINAL_AI/mo_hinh_banh_3I_v3.keras')
print("Đã lưu model phiên bản mới (V2) an toàn vào Drive!")

Đã lưu model phiên bản mới (V2) an toàn vào Drive!


In [ ]:
import cv2
import numpy as np
import gradio as gr
import traceback
from tensorflow.keras.models import load_model

print("Đang nạp mô hình AI...")
model = load_model('/content/drive/MyDrive/FINAL_AI/sweetcheckout3final.keras')

try:
    _, h_size, w_size, _ = model.input_shape
    IMG_SIZE = (w_size, h_size)
except:
    IMG_SIZE = (128, 128)

CLASS_NAMES = [
    "banana_cake", "banh_da_lon", "blueberry_muffin", "butter_bread",
    "cha_bong_cay", "coconut_cookies", "croissant", "eggtart",
    "melon_bread", "patechaud"
]

GIA_BANH = {
    "banana_cake": {"ten": "Bánh chuối nướng", "ten_khong_dau": "Banh chuoi nuong", "gia": 19000},
    "banh_da_lon": {"ten": "Bánh da lợn", "ten_khong_dau": "Banh da lon", "gia": 23000},
    "blueberry_muffin": {"ten": "Muffin việt quất", "ten_khong_dau": "Muffin viet quat", "gia": 25000},
    "butter_bread": {"ten": "Bánh mì bơ", "ten_khong_dau": "Banh mi bo", "gia": 18000},
    "cha_bong_cay": {"ten": "Chà bông cây", "ten_khong_dau": "Cha bong cay", "gia": 27000},
    "coconut_cookies": {"ten": "Cookies dừa", "ten_khong_dau": "Cookies dua", "gia": 23000},
    "croissant": {"ten": "Croissant", "ten_khong_dau": "Croissant", "gia": 30000},
    "eggtart": {"ten": "Egg Tart", "ten_khong_dau": "Egg Tart", "gia": 21000},
    "melon_bread": {"ten": "Bánh mì dừa lưới", "ten_khong_dau": "Banh mi dua luoi", "gia": 15000},
    "patechaud": {"ten": "Patechaud", "ten_khong_dau": "Patechaud", "gia": 30000}
}

def xu_ly_khay_banh(image):
    try:
        if image is None: return None, "Vui lòng tải ảnh!"

        img = np.array(image)
        if len(img.shape) == 3 and img.shape[2] == 4: img = img[:, :, :3]

        h, w = img.shape[:2]

        rois = {
            "Top_Trai": (0.02, 0.02, 0.46, 0.46),
            "Top_Phai": (0.52, 0.02, 0.46, 0.46),
            "Bot_Trai": (0.02, 0.52, 0.30, 0.46),
            "Bot_Giua": (0.35, 0.52, 0.30, 0.46),
            "Bot_Phai": (0.68, 0.52, 0.30, 0.46)
        }

        img_draw = img.copy()
        tong_tien = 0

        hoadon_html = """
        <div style='background:#fffaf6; border:2px dashed #ff9a8b; border-radius:10px; padding:20px; color:#4a2c1d;'>
            <h2 style='text-align:center; color:#d6336c; margin-top:0;'>🧾 HÓA ĐƠN SWEETCHECKOUT</h2>
            <hr style='border-top: 1px dashed #ff9a8b;'>
            <table style='width:100%; font-size:16px; color:#4a2c1d;'>
        """

        for key, (x_pct, y_pct, w_pct, h_pct) in rois.items():
            x, y = int(x_pct * w), int(y_pct * h)
            roi_w, roi_h = int(w_pct * w), int(h_pct * h)

            crop_img = img[y : y + roi_h, x : x + roi_w]
            piece_resized = cv2.resize(crop_img, IMG_SIZE)

            # CHUẨN TRẢ VỀ /255.0 CHO MẠNG TỰ XÂY
            piece_arr = np.array(piece_resized).astype('float32') / 255.0
            piece_arr = np.expand_dims(piece_arr, axis=0)

            preds = model.predict(piece_arr, verbose=0)[0]
            idx = int(np.argmax(preds))

            thong_tin = GIA_BANH[CLASS_NAMES[idx]]
            gia = thong_tin["gia"]
            tong_tien += gia

            cv2.rectangle(img_draw, (x, y), (x + roi_w, y + roi_h), (0, 255, 0), 4)
            label = f'{thong_tin["ten_khong_dau"]} ({gia/1000:g}k) [{preds[idx]*100:.1f}%]'

            cv2.putText(img_draw, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, w / 1100, (0, 0, 0), 4, cv2.LINE_AA)
            cv2.putText(img_draw, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, w / 1100, (255, 0, 0), 2, cv2.LINE_AA)

            if gia > 0:
                hoadon_html += f"<tr><td style='padding:5px 0; border-bottom: 1px solid #f1e3da;'>🍰 {thong_tin['ten']}</td><td style='text-align:right; font-weight:bold; border-bottom: 1px solid #f1e3da;'>{gia:,} VNĐ</td></tr>"

        hoadon_html += f"</table><h2 style='text-align:right; color:#d6336c; margin-top:15px;'>TỔNG CỘNG: {tong_tien:,} VNĐ</h2></div>"
        return img_draw, hoadon_html

    except Exception as e:
        return image, f"Lỗi: {str(e)} - {traceback.format_exc()}"

GIA_BANH_THEME = gr.themes.Soft(primary_hue='pink', secondary_hue='orange')
with gr.Blocks(theme=GIA_BANH_THEME) as demo:
    gr.HTML("<div style='text-align:center; padding:15px; background:linear-gradient(135deg, #ff9a8b, #ff6a88); border-radius:10px; color:white;'><h1 style='margin:0; font-size:28px;'>SWEETCHECKOUT DEMO</h1></div>")
    with gr.Row():
        with gr.Column(scale=6):
            anh_dau_vao = gr.Image(sources=['upload', 'webcam'], type='numpy', label='📷 Tải ảnh khay bánh (Chụp ngang)')
            nut_nhan_dien = gr.Button('🔍 XUẤT HÓA ĐƠN', variant='primary', size='lg')
        with gr.Column(scale=4):
            ket_qua_hoadon = gr.HTML("Hóa đơn...")
    anh_dau_ra = gr.Image(label='Kết quả AI phân tích', interactive=False)
    nut_nhan_dien.click(fn=xu_ly_khay_banh, inputs=anh_dau_vao, outputs=[anh_dau_ra, ket_qua_hoadon])

demo.launch(share=True, debug=True)

Đang nạp mô hình AI...


/tmp/ipykernel_562/2586870699.py:96: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=GIA_BANH_THEME) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://53052406e9eb69e8a8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://53052406e9eb69e8a8.gradio.live


In [ ]:
import cv2
import numpy as np
import gradio as gr
import traceback
from tensorflow.keras.models import load_model

print("Đang nạp mô hình AI...")
model = load_model('/content/drive/MyDrive/FINAL_AI/sweetcheckout3final.keras')

try:
    _, h_size, w_size, _ = model.input_shape
    IMG_SIZE = (w_size, h_size)
except:
    IMG_SIZE = (128, 128)

CLASS_NAMES = [
    "banana_cake", "banh_da_lon", "blueberry_muffin", "butter_bread",
    "cha_bong_cay", "coconut_cookies", "croissant", "eggtart",
    "melon_bread", "patechaud"
]

GIA_BANH = {
    "banana_cake": {"ten": "Bánh chuối nướng", "ten_khong_dau": "Banh chuoi nuong", "gia": 19000},
    "banh_da_lon": {"ten": "Bánh da lợn", "ten_khong_dau": "Banh da lon", "gia": 23000},
    "blueberry_muffin": {"ten": "Muffin việt quất", "ten_khong_dau": "Muffin viet quat", "gia": 25000},
    "butter_bread": {"ten": "Bánh mì bơ", "ten_khong_dau": "Banh mi bo", "gia": 18000},
    "cha_bong_cay": {"ten": "Chà bông cây", "ten_khong_dau": "Cha bong cay", "gia": 27000},
    "coconut_cookies": {"ten": "Cookies dừa", "ten_khong_dau": "Cookies dua", "gia": 23000},
    "croissant": {"ten": "Croissant", "ten_khong_dau": "Croissant", "gia": 30000},
    "eggtart": {"ten": "Egg Tart", "ten_khong_dau": "Egg Tart", "gia": 21000},
    "melon_bread": {"ten": "Bánh mì dừa lưới", "ten_khong_dau": "Banh mi dua luoi", "gia": 15000},
    "patechaud": {"ten": "Patechaud", "ten_khong_dau": "Patechaud", "gia": 30000}
}

import cv2
import numpy as np
import gradio as gr
import traceback
from tensorflow.keras.models import load_model

print("Đang nạp mô hình AI...")
model = load_model('/content/drive/MyDrive/FINAL_AI/sweetcheckout3final.keras')

try:
    _, h_size, w_size, _ = model.input_shape
    IMG_SIZE = (w_size, h_size)
except:
    IMG_SIZE = (128, 128)

CLASS_NAMES = [
    "banana_cake", "banh_da_lon", "blueberry_muffin", "butter_bread",
    "cha_bong_cay", "coconut_cookies", "croissant", "eggtart",
    "melon_bread", "patechaud"
]

GIA_BANH = {
    "banana_cake": {"ten": "Bánh chuối nướng", "ten_khong_dau": "Banh chuoi nuong", "gia": 19000},
    "banh_da_lon": {"ten": "Bánh da lợn", "ten_khong_dau": "Banh da lon", "gia": 23000},
    "blueberry_muffin": {"ten": "Muffin việt quất", "ten_khong_dau": "Muffin viet quat", "gia": 25000},
    "butter_bread": {"ten": "Bánh mì bơ", "ten_khong_dau": "Banh mi bo", "gia": 18000},
    "cha_bong_cay": {"ten": "Chà bông cây", "ten_khong_dau": "Cha bong cay", "gia": 27000},
    "coconut_cookies": {"ten": "Cookies dừa", "ten_khong_dau": "Cookies dua", "gia": 23000},
    "croissant": {"ten": "Croissant", "ten_khong_dau": "Croissant", "gia": 30000},
    "eggtart": {"ten": "Egg Tart", "ten_khong_dau": "Egg Tart", "gia": 21000},
    "melon_bread": {"ten": "Bánh mì dừa lưới", "ten_khong_dau": "Banh mi dua luoi", "gia": 15000},
    "patechaud": {"ten": "Patechaud", "ten_khong_dau": "Patechaud", "gia": 30000}
}

def xu_ly_khay_banh(image):
    try:
        if image is None: return None, "Vui lòng tải ảnh!"

        img = np.array(image)
        if len(img.shape) == 3 and img.shape[2] == 4: img = img[:, :, :3]

        h, w = img.shape[:2]

        rois = {
            "Top_Trai": (0.02, 0.02, 0.46, 0.46),
            "Top_Phai": (0.52, 0.02, 0.46, 0.46),
            "Bot_Trai": (0.02, 0.52, 0.30, 0.46),
            "Bot_Giua": (0.35, 0.52, 0.30, 0.46),
            "Bot_Phai": (0.68, 0.52, 0.30, 0.46)
        }

        img_draw = img.copy()
        tong_tien = 0

        hoadon_html = """
        <div style='background:#fffaf6; border:2px dashed #ff9a8b; border-radius:10px; padding:20px; color:#4a2c1d;'>
            <h2 style='text-align:center; color:#d6336c; margin-top:0;'>🧾 HÓA ĐƠN SWEETCHECKOUT</h2>
            <hr style='border-top: 1px dashed #ff9a8b;'>
            <table style='width:100%; font-size:16px; color:#4a2c1d;'>
        """

        for key, (x_pct, y_pct, w_pct, h_pct) in rois.items():
            x, y = int(x_pct * w), int(y_pct * h)
            roi_w, roi_h = int(w_pct * w), int(h_pct * h)

            crop_img = img[y : y + roi_h, x : x + roi_w]
            piece_resized = cv2.resize(crop_img, IMG_SIZE)

            # CHUẨN TRẢ VỀ /255.0 CHO MẠNG TỰ XÂY
            piece_arr = np.array(piece_resized).astype('float32') / 255.0
            piece_arr = np.expand_dims(piece_arr, axis=0)

            preds = model.predict(piece_arr, verbose=0)[0]
            idx = int(np.argmax(preds))

            thong_tin = GIA_BANH[CLASS_NAMES[idx]]
            gia = thong_tin["gia"]
            tong_tien += gia

            cv2.rectangle(img_draw, (x, y), (x + roi_w, y + roi_h), (0, 255, 0), 4)
            label = f'{thong_tin["ten_khong_dau"]} ({gia/1000:g}k) [{preds[idx]*100:.1f}%]'

            cv2.putText(img_draw, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, w / 1100, (0, 0, 0), 4, cv2.LINE_AA)
            cv2.putText(img_draw, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, w / 1100, (255, 0, 0), 2, cv2.LINE_AA)

            if gia > 0:
                hoadon_html += f"<tr><td style='padding:5px 0; border-bottom: 1px solid #f1e3da;'>🍰 {thong_tin['ten']}</td><td style='text-align:right; font-weight:bold; border-bottom: 1px solid #f1e3da;'>{gia:,} VNĐ</td></tr>"

        hoadon_html += f"</table><h2 style='text-align:right; color:#d6336c; margin-top:15px;'>TỔNG CỘNG: {tong_tien:,} VNĐ</h2></div>"
        return img_draw, hoadon_html

    except Exception as e:
        return image, f"Lỗi: {str(e)} - {traceback.format_exc()}"

GIA_BANH_THEME = gr.themes.Soft(primary_hue='pink', secondary_hue='orange')
with gr.Blocks(theme=GIA_BANH_THEME) as demo:
    gr.HTML("<div style='text-align:center; padding:15px; background:linear-gradient(135deg, #ff9a8b, #ff6a88); border-radius:10px; color:white;'><h1 style='margin:0; font-size:28px;'>SWEETCHECKOUT DEMO</h1></div>")
    with gr.Row():
        with gr.Column(scale=6):
            anh_dau_vao = gr.Image(sources=['upload', 'webcam'], type='numpy', label='📷 Tải ảnh khay bánh (Chụp ngang)')
            nut_nhan_dien = gr.Button('🔍 XUẤT HÓA ĐƠN', variant='primary', size='lg')
        with gr.Column(scale=4):
            ket_qua_hoadon = gr.HTML("Hóa đơn...")
    anh_dau_ra = gr.Image(label='Kết quả AI phân tích', interactive=False)
    nut_nhan_dien.click(fn=xu_ly_khay_banh, inputs=anh_dau_vao, outputs=[anh_dau_ra, ket_qua_hoadon])

demo.launch(share=True, debug=True)
GIA_BANH_THEME = gr.themes.Soft(primary_hue='pink', secondary_hue='orange')
with gr.Blocks(theme=GIA_BANH_THEME) as demo:
    gr.HTML("<div style='text-align:center; padding:15px; background:linear-gradient(135deg, #ff9a8b, #ff6a88); border-radius:10px; color:white;'><h1 style='margin:0; font-size:28px;'>SWEETCHECKOUT DEMO</h1></div>")
    with gr.Row():
        with gr.Column(scale=6):
            anh_dau_vao = gr.Image(sources=['upload', 'webcam'], type='numpy', label='📷 Tải ảnh khay bánh (Chụp ngang)')
            nut_nhan_dien = gr.Button('🔍 XUẤT HÓA ĐƠN', variant='primary', size='lg')
        with gr.Column(scale=4):
            ket_qua_hoadon = gr.HTML("Hóa đơn...")
    anh_dau_ra = gr.Image(label='Kết quả AI phân tích', interactive=False)
    nut_nhan_dien.click(fn=xu_ly_khay_banh, inputs=anh_dau_vao, outputs=[anh_dau_ra, ket_qua_hoadon])

demo.launch(share=True, debug=True)

Đang nạp mô hình AI...
Đang nạp mô hình AI...


/tmp/ipykernel_562/3667062504.py:130: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=GIA_BANH_THEME) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://50480cf2876ac518b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error